# 02 — Bridging the two plates of PAD000040

**Dataset:** *Inflammatory proteomic heterogeneity beyond glycemia status in severe obesity.*
Milito MM, ..., Giraudi PJ. Int J Mol Sci 2026;27(9):4152. doi:10.3390/ijms27094152.
PRIDE: [PAD000040](https://www.ebi.ac.uk/pride/archive/projects/PAD000040) — licence CC0 (public domain).

**Why this notebook.** The study was measured on two plates, a year apart (2024 and 2025). Notebook 01
found that plate 2 reads higher, and that **prediabetes was only measured on plate 2** (and among the
bridging samples). Without bridging, any "prediabetes difference" could simply be the plate.

**The idea of bridging.** Some tubes were measured on *both* plates on purpose: the **bridging
samples**. Same tube → same biology, so any difference between its two values is technical. Per
protein, the **median** difference over the bridging samples is the plate shift; we subtract it from
plate 2. Plate 1 (the reference) stays as it is.

**Steps**
1. Read both plate files and the sample list; quality steps.
2. **Look** at the plate difference (`bridge_check`, charts) — change nothing.
3. Deal with one broken bridging measurement.
4. **Bridge** (`bridge_plates`).
5. **Check** that it worked — including a check that does not reuse the bridging numbers (PCA).

In [1]:
from pathlib import Path
import pandas as pd

from signatomics.olink import read_npx, add_sample_info, apply_qc, plot_sample_qc   # our package
from signatomics.olink import (bridge_check, bridge_plates,
                               plot_bridge_shift, plot_bridge_scatter, plot_bridge_pca)

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

# Find the signatomics folder, wherever the notebook is started from. Data: data/PAD000040/ (never in git).
here = Path.cwd()
ROOT = next(p for p in [here, *here.parents, here / "signatomics"] if (p / "src" / "signatomics").is_dir())
DATA = ROOT / "data" / "PAD000040"

## 1. Read the two plates, attach the sample list, quality steps

We use the two **separate plate files** (`*.raw.csv`), not the authors' combined file — notebook 01
showed that the combined file still contains the plate difference. `pd.concat` puts the two plates in
one table; `PlateID` tells them apart.

In [2]:
plates = pd.concat([read_npx(DATA / "inflam_NASH_dt1.raw.csv"),
                    read_npx(DATA / "inflam_PRA_dt2.raw.csv")], ignore_index=True)
data = add_sample_info(plates, DATA / "2025.05.08_sample_manifest.xlsx", on="FIF ID")

Reading report
  file: inflam_NASH_dt1.raw.csv
  file type: text, separated by comma
  layout: long (one row per sample and protein)
  standard column names: PlateLOD -> LOD
  MissingFreq: converted percentages
  result: 8280 rows, 90 sample IDs, 92 proteins, 90 sample measurements, 1 plates
Reading report
  file: inflam_PRA_dt2.raw.csv
  file type: text, separated by comma
  layout: long (one row per sample and protein)
  standard column names: PlateLOD -> LOD
  NPX: converted 184 values written as No Data -> empty
  MissingFreq: converted percentages
  result: 8096 rows, 88 sample IDs, 92 proteins, 88 sample measurements, 1 plates


Matching report
  protein data: 143 sample IDs; sample list: 142 IDs (matched on 'SampleID' = 'FIF ID')
  matched: 142
  WARNING in protein data but NOT in the sample list (kept, without sample information): 249
  columns that exist in both files got the ending _samplelist: PlateID


**Note the matching report:** sample **249** is in the plate files but *not* in the authors' sample
list. Keep it in mind — it comes back in section 3.

In [3]:
clean = apply_qc(data, show_ids=False)

Quality steps applied
  removed 2 measurements (no values): 2 (IDs hidden)
  removed 15 measurements (quality warning): 14 (IDs hidden)
  kept: 161 measurements, 132 sample IDs, 92 proteins


## 2. Look at the plate difference — before changing anything

`bridge_check` compares, for every bridging sample, its value on plate 2 with its value on plate 1
(the reference: the plate with most samples). With `group="diabetes"` it also checks whether the
bridging samples cover every group, and whether the shift is the same in each group.

In [4]:
before = bridge_check(clean, group="diabetes", show_ids=False)

Bridging check
  reference: Inflammation F.I.F 21/03/2024
  Inflammation PRAESIIDIUM Giraudi 04_03_2025: 29 bridging samples; reads +0.52 NPX vs the reference (median over proteins; per protein -0.96 to +2.30; 1.44 times)
    proteins shifted by more than 0.1 NPX: 83 of 92 - bridging needed
    noise: the bridge differences of one protein typically spread 0.44 NPX (IQR) - measurement noise; bridging removes the shift, not this
    bridging samples that shift differently from the others (check them: mix-up or degraded tube?): 2 (IDs hidden)
  'diabetes' among the bridging samples: No diabetes: 9 of 62; T2DM: 9 of 46; Prediabetes: 10 of 23
    overall shift on Inflammation PRAESIIDIUM Giraudi 04_03_2025 per group: No diabetes +0.56; Prediabetes +0.57; T2DM +0.53 (largest difference 0.04 NPX)
    proteins whose shift differs by more than 0.5 NPX between groups: IL13, IL-22 RA1, TNFB, 4E-BP1, IL-24, IL2 - with few bridges per group this is often noise; look at them before trusting group di

**How to read this:**
- Plate 2 reads **about +0.5 NPX higher** (≈1.4 times) — but **per protein the shift ranges from
  about −1 to +2.3 NPX**. One shift for all proteins would be wrong: bridging must be per protein.
- **The shift is the same in every diabetes group** (overall about +0.53 to +0.57). That is the most
  important result for this study: the plate shift does not depend on diabetes status, so removing it
  does not remove or create a diabetes difference.
- All three groups are among the bridging samples (prediabetes: 10 of 23), so the check is possible.
- A few proteins have a different shift per group; with ~10 bridging samples per group that is
  usually noise, but we note them.
- **Two bridging samples shift differently from the others** — section 3.

In [5]:
plot_bridge_shift(clean)

**How to read this chart:** each grey dot is one bridging sample (plate 2 minus plate 1); the orange
square is the median (the shift that bridging will remove), the bar the middle 50%. Most squares sit
right of 0 (plate 2 higher), by different amounts per protein.

**One dot keeps appearing far to the left (−5 to −15 NPX) for almost every protein.** That is one
bridging sample whose plate-2 values are far too low across the board.

In [6]:
plot_bridge_scatter(clean, ["IL8", "MCP-1", "IL6", "CCL4", "IL-18R1", "TNFB"])

Same tube on two plates: plate 1 on x, plate 2 on y. On the dashed line both plates agree. The cloud
lies *above* the line (plate 2 higher) — and one point lies far below, near or under the detection
limit (dotted) on plate 2 for every protein.

## 3. One broken bridging measurement

Which sample is it? `bridge_check` keeps the per-sample shifts in `.samples`:

In [7]:
shifts = before.samples.sort_values("shift")
pd.concat([shifts.head(2), shifts.tail(2)])    # the lowest and the highest

,plate,sample,shift
13,Inflammation PRAESIIDIUM Giraudi 04_03_2025,249,-5.238875
21,Inflammation PRAESIIDIUM Giraudi 04_03_2025,51,-0.080180
9,Inflammation PRAESIIDIUM Giraudi 04_03_2025,219,1.132995
3,Inflammation PRAESIIDIUM Giraudi 04_03_2025,143,1.665190


In [8]:
plot_sample_qc(clean)

It is **sample 249 on plate 2**: on average 5 NPX (≈ 40 times) lower than on plate 1, and the
sample check above flags it as an outlier (lowest median, far outside the dashed lines) — although it
**passed Olink's own quality check**. It is also the sample missing from the authors' sample list and
from their combined file: **the authors removed it too.** Three independent signals agree.

The median makes the bridging robust to one broken sample, but this measurement should not be analysed.
`apply_qc(drop_samples=...)` removes it with a report; a dict removes it only on one plate.

The other flagged sample, **143**, reads about 1.7 NPX higher on plate 2 — roughly 1 NPX more than the
typical shift. That is much milder: it passed quality control on both plates and is not an outlier in
the sample check. We **keep** it; the median keeps a single sample from pulling the shift.

In [9]:
clean = apply_qc(data, drop_samples={"249": "Inflammation PRAESIIDIUM Giraudi 04_03_2025"}, show_ids=False)

Quality steps applied
  removed 1 measurements (chosen by you (drop_samples)): 1 (IDs hidden)
  removed 2 measurements (no values): 2 (IDs hidden)
  removed 15 measurements (quality warning): 14 (IDs hidden)
  kept: 160 measurements, 132 sample IDs, 92 proteins


## 4. Bridge

First the factors — the shift per protein — as a table you can inspect or save
(`factors.to_csv(...)`). `adjustment` is what is added to plate 2; `spread` is how much the bridging
samples disagree (IQR) — noise that bridging cannot remove.

In [10]:
factors = bridge_check(clean, verbose=False).factors
factors.sort_values("adjustment").head(8)

,plate,reference,OlinkID,Assay,adjustment,bridges,spread,few_bridges
29,Inflammation PRAESIIDIUM Giraudi 04_03_2025,Inflammation F.I.F 21/03/2024,OID01213,DNER,-2.307230,28,0.314910,False
75,Inflammation PRAESIIDIUM Giraudi 04_03_2025,Inflammation F.I.F 21/03/2024,OID00518,PD-L1,-1.971905,28,0.371515,False
11,Inflammation PRAESIIDIUM Giraudi 04_03_2025,Inflammation F.I.F 21/03/2024,OID00539,CCL28,-1.864575,28,0.502542,False
9,Inflammation PRAESIIDIUM Giraudi 04_03_2025,Inflammation F.I.F 21/03/2024,OID00530,CCL23,-1.792820,28,0.475702,False
62,Inflammation PRAESIIDIUM Giraudi 04_03_2025,Inflammation F.I.F 21/03/2024,OID00480,LAP TGF-beta-1,-1.482405,28,0.555880,False
46,Inflammation PRAESIIDIUM Giraudi 04_03_2025,Inflammation F.I.F 21/03/2024,OID00517,IL-18R1,-1.420190,28,0.379820,False
30,Inflammation PRAESIIDIUM Giraudi 04_03_2025,Inflammation F.I.F 21/03/2024,OID00541,EN-RAGE,-1.294775,28,0.601635,False
25,Inflammation PRAESIIDIUM Giraudi 04_03_2025,Inflammation F.I.F 21/03/2024,OID00486,CXCL11,-1.289950,28,0.363602,False


In [11]:
bridged = bridge_plates(clean, factors=factors)

Bridging report
  reference (not changed): Inflammation F.I.F 21/03/2024
  Inflammation PRAESIIDIUM Giraudi 04_03_2025: 92 proteins adjusted, median -0.55 NPX (from -2.31 to +0.93); bridging samples per protein: 28-28
  NPX, LOD and MaxLOD shifted together; originals in NPX_unbridged / LOD_unbridged
  next: bridge_check(...) on the result - the shifts should now be about 0


`bridge_plates` shifts NPX **and the detection limits** (LOD, MaxLOD) of plate 2 together, so a value
below the detection limit stays below it. The original values stay in `NPX_unbridged`.

In [12]:
bridged.loc[bridged["Assay"] == "IL8", ["SampleID", "PlateID", "NPX_unbridged", "Bridge_adjustment", "NPX", "LOD"]].head(4)

,SampleID,PlateID,NPX_unbridged,Bridge_adjustment,NPX,LOD
0,58,Inflammation F.I.F 21/03/2024,5.11826,0.0,5.11826,0.316384
92,301,Inflammation F.I.F 21/03/2024,4.95001,0.0,4.95001,0.316384
184,177,Inflammation F.I.F 21/03/2024,5.40016,0.0,5.40016,0.316384
276,37,Inflammation F.I.F 21/03/2024,6.32382,0.0,6.32382,0.316384


## 5. Did it work?

**Check 1 — the bridging samples themselves.** Shifts should now be about 0. But be honest about what
this shows: the adjustment was *computed* from these same samples, so ~0 is guaranteed. It confirms
the step ran correctly, not that bridging was a good idea.

In [13]:
after = bridge_check(bridged, group="diabetes", show_ids=False)

Bridging check
  reference: Inflammation F.I.F 21/03/2024
  Inflammation PRAESIIDIUM Giraudi 04_03_2025: 28 bridging samples; reads +0.00 NPX vs the reference (median over proteins; per protein +0.00 to +0.00; 1.00 times)
    proteins shifted by more than 0.1 NPX: 0 of 92 - the plates are lined up
    noise: the bridge differences of one protein typically spread 0.42 NPX (IQR) - measurement noise; bridging removes the shift, not this
    bridging samples that shift differently from the others (check them: mix-up or degraded tube?): 2 (IDs hidden)
  note: this table is already bridged with these bridging samples, so shifts of about 0 are expected by construction. Independent checks: plot_bridge_pca (the same samples should now overlap) and, with group=..., whether the shift was the same in every group
  'diabetes' among the bridging samples: No diabetes: 9 of 62; T2DM: 9 of 46; Prediabetes: 10 of 23
    overall shift on Inflammation PRAESIIDIUM Giraudi 04_03_2025 per group: No diabetes 

In [14]:
plot_bridge_shift(bridged)

**Check 2 — independent: all proteins together (PCA).** Bridging was fitted *one protein at a time*.
A PCA looks at all 92 proteins together. Each bridging sample is two points (one per plate) joined by a
thin line. Before bridging the plates form two clouds; after bridging they should overlap and the lines
should shrink.

In [15]:
plot_bridge_pca(clean, bridged)

**Check 3 — compare with the authors' combined file.** Their file is meant to be bridged already; we
run the same check on it.

In [16]:
authors = read_npx(DATA / "dataset_all.npx.csv", verbose=False)
_ = bridge_check(apply_qc(authors, verbose=False), show_ids=False)

Bridging check
  reference: Inflammation F.I.F 21/03/2024
  Inflammation PRAESIIDIUM Giraudi 04_03_2025: 28 bridging samples; reads +0.54 NPX vs the reference (median over proteins; per protein -0.91 to +2.26; 1.45 times)
    proteins shifted by more than 0.1 NPX: 84 of 92 - bridging needed
    noise: the bridge differences of one protein typically spread 0.41 NPX (IQR) - measurement noise; bridging removes the shift, not this
    bridging samples that shift differently from the others (check them: mix-up or degraded tube?): 1 (IDs hidden)


## Summary

- The two plates of PAD000040 differ by **about +0.5 NPX on plate 2, but differently per protein**
  (−1 to +2.3 NPX) — bridging per protein is needed.
- The shift is the **same in every diabetes group**, so bridging does not distort the diabetes
  comparison — the key worry from notebook 01.
- One bridging measurement (**sample 249, plate 2**) is broken despite passing Olink QC; three
  independent signals flagged it, and the authors removed it as well. We removed it with
  `apply_qc(drop_samples=...)`.
- After bridging the plates overlap in the PCA (a sample's two measurements: 8.6 → 3.0 apart). The authors' combined file still has the plate
  difference; **from now on we use `bridged`**, built from the raw plate files.

**Next:** decide what to do with the bridging samples' *two* measurements per person (average them so
each person has one value per protein), then the first group comparisons and the biomarker signature.